# 01 — Dữ liệu và PIT

Chứng minh đúng input, config, code và các mẫu financial state theo ngày quyết định. Không fit mô hình, không thu thập lại dữ liệu.

**Đầu vào:** `M3/sources.json`, hai run/seal Dev và Holdout, bảy input đã pin. **Đầu ra:** các bảng dưới đây và `01_verification.json` trong thư mục reproduction.

Hash sai, source thay đổi hoặc target mẫu không khớp sẽ dừng bằng assertion. Các cell bên dưới chứa phần điều phối; quy tắc PIT/strategy nằm trong module chính thức.

### Thiết lập trực tiếp trong notebook

Chọn kernel Python có các phụ thuộc trong lock, rồi **Run All**. Các cell dưới đọc pin và định nghĩa phần điều phối ngay trong notebook; không import helper Python từ `M3`. Logic nghiên cứu vẫn gọi module chính thức trong `src/delta_t1/m3` và hàm metrics của `scripts/run_m3.py`.

In [1]:
"""Điều phối tái lập M3-C; mọi PIT/ranking/backtest gọi module M3 chính thức."""
from __future__ import annotations

from bisect import bisect_right
from collections import Counter, defaultdict
from datetime import datetime, timezone
import hashlib
import html
import importlib.util
import json
import math
import os
from pathlib import Path
import csv
import subprocess
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'M3/sources.json').is_file())
runtime = ROOT / '.tmp/m3c-notebook-runtime'
if runtime.is_dir():
    sys.path.insert(0, str(runtime))
sys.path.insert(0, str(ROOT / 'src'))
from delta_t1.m3 import inputs, ranking, universe
from delta_t1.m3.benchmark import benchmark_nav
from delta_t1.m3.engine import run as portfolio_run
from delta_t1.m3.prices import PriceBook
from delta_t1.m3.strategies import Strategies

PIN = json.loads((ROOT / 'M3/sources.json').read_text(encoding='utf-8'))
spec = importlib.util.spec_from_file_location('m3_official_runner', ROOT / 'scripts/run_m3.py')
official = importlib.util.module_from_spec(spec)
spec.loader.exec_module(official)
NAMES = dict(combined='Combined', market_only='Market-only', financial_only='Financial-only',
             equal_weight='Equal-weight', momentum_only='Momentum-only', vnindex='VNINDEX')
RTOL, ATOL = 1e-9, 1e-10

### Đọc dữ liệu và đối chiếu

Identifiers/schema/order phải khớp chính xác; số thực theo tolerance của evidence.

In [2]:
def sha(path):
    return inputs.sha256(path)

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def write_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

def csv_rows(path):
    with Path(path).open(encoding='utf-8-sig', newline='') as file:
        return list(csv.DictReader(file))

def grouped(rows, key):
    result = defaultdict(list)
    for row in rows:
        result[row[key]].append(row)
    return dict(result)

def equal(actual, expected, context=''):
    if isinstance(actual, bool):
        assert str(actual) == str(expected), (context, actual, expected)
    elif isinstance(actual, (float, int)):
        assert math.isfinite(float(expected)) and math.isclose(float(actual), float(expected), rel_tol=RTOL, abs_tol=ATOL), (context, actual, expected)
    elif actual is None:
        assert expected in (None, ''), (context, actual, expected)
    else:
        assert str(actual) == str(expected), (context, actual, expected)

def compare_rows(actual, expected, context):
    assert len(actual) == len(expected), (context, len(actual), len(expected))
    for index, (left, right) in enumerate(zip(actual, expected)):
        assert set(left) == set(right), (context, index, set(left) ^ set(right))
        for key, value in left.items():
            equal(value, right[key], (context, index, key))

def table(rows, columns=None):
    """Bảng HTML hiển thị trong notebook, không sinh canonical CSV trung gian."""
    from IPython.display import HTML, display
    columns = list(columns or (list(rows[0]) if rows else []))
    def cell(value):
        if value is None:
            return ''
        if isinstance(value, float):
            return f'{value:.8g}'
        return str(value)
    head = ''.join(f'<th>{html.escape(c)}</th>' for c in columns)
    body = ''.join('<tr>' + ''.join(f'<td>{html.escape(cell(row.get(c)))}</td>' for c in columns) + '</tr>' for row in rows)
    display(HTML('<div style="overflow:auto"><table style="border-collapse:collapse;font-size:13px">'
                 '<thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table></div>'))

def notebook_code_sha256(path):
    notebook = read_json(path)
    pieces = [''.join(c['source']) for c in notebook['cells'] if c['cell_type'] == 'code']
    return hashlib.sha256('\n\n'.join(pieces).encode('utf-8')).hexdigest()

### Pin nguồn và kiểm tra input

Hash bảy input; lấy trần/sàn trong cùng lượt đọc C8. Nếu các notebook cùng phiên dùng `M3C_OUTPUT_DIR`, proof được kiểm tra fingerprint và tái sử dụng.

In [3]:
def source_identity():
    scope = ['src/delta_t1/m3', 'scripts/run_m3.py', 'configs/experiments/m3_strategy_v1.json']
    assert not subprocess.check_output(['git', 'diff', PIN['source_commit'], '--', *scope], cwd=ROOT), 'M3 source khác commit đã pin'
    assert not subprocess.check_output(['git', 'status', '--porcelain', '--', *scope], cwd=ROOT), 'M3 source/config chưa sạch'
    return dict(source_commit=PIN['source_commit'],
                execution_commit=subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=ROOT, text=True).strip(),
                branch=subprocess.check_output(['git', 'branch', '--show-current'], cwd=ROOT, text=True).strip(),
                worktree_status=subprocess.check_output(['git', 'status', '--short'], cwd=ROOT, text=True).strip(),
                python=sys.version.split()[0], executable=sys.executable,
                module_hashes={str(p.relative_to(ROOT)).replace('\\', '/'): sha(p)
                               for p in [*sorted((ROOT / 'src/delta_t1/m3').glob('*.py')), ROOT / 'scripts/run_m3.py']})

def input_paths(config):
    return dict(terminal_sha256=inputs.BUNDLE, indicators_sha256=inputs.INDICATORS,
                availability_sha256=inputs.AVAILABILITY, c8_benchmark_sha256=inputs.BENCHMARK,
                c8_calendar_sha256=inputs.CALENDAR, c8_prices_sha256=inputs.PRICES_C8,
                b_pit_labels_sha256=ROOT / config['financial_pit']['artifact'] / 'labels.csv')

def fingerprint(path):
    stat = path.stat()
    return dict(size=stat.st_size, mtime_ns=stat.st_mtime_ns)

def validate_runs():
    config = read_json(ROOT / PIN['config'])
    assert config['status'] == 'LOCKED' and sha(ROOT / PIN['config']) == PIN['config_sha256']
    runs = {}
    for segment, pin in PIN['runs'].items():
        seal_path = ROOT / pin['seal']
        assert sha(seal_path) == pin['seal_sha256'], segment
        seal = read_json(seal_path)
        folder = ROOT / seal['run']
        manifest = read_json(folder / 'manifest.json')
        assert seal['status'] == 'PASS' and len(seal['artifacts_sha256']) == 33
        assert {p.name for p in folder.iterdir() if p.is_file()} == set(seal['artifacts_sha256'])
        for name, digest in seal['artifacts_sha256'].items():
            assert sha(folder / name) == digest, (segment, name)
        for key in ['git_commit', 'config_sha256', 'inputs_sha256']:
            assert manifest[key] == seal[key]
        assert manifest['git_commit'] == PIN['source_commit']
        assert manifest['config_sha256'] == PIN['config_sha256']
        assert manifest['financial_source'] == 'B_PIT' and not manifest['targets_only'] and not manifest['git_dirty']
        assert manifest['segment'] == segment and manifest['status'] == 'LOCKED'
        runs[segment] = dict(folder=folder, manifest=manifest, seal=seal,
                             metrics=read_json(folder / 'metrics.json'))
    assert runs['development']['manifest']['inputs_sha256'] == runs['holdout']['manifest']['inputs_sha256']
    return config, runs

def preflight(output):
    """Hash đủ input; cùng lượt đọc C8 lưu trần/sàn cho hai segment để tránh scan lặp."""
    output = Path(output).resolve()
    output.mkdir(parents=True, exist_ok=True)
    identity = source_identity()
    config, runs = validate_runs()
    expected = runs['development']['manifest']['inputs_sha256']
    dates = {r['date'] for run in runs.values() for r in csv_rows(run['folder'] / 'nav_combined.csv')}
    hashes = {}; limits = []
    paths = input_paths(config)
    for key, path in paths.items():
        before = fingerprint(path)
        if key == 'c8_prices_sha256':
            digest = hashlib.sha256(); seen = set()
            with path.open('rb') as file:
                for line in file:
                    digest.update(line)
                    marker = b'"trade_date": "'
                    pos = line.index(marker) + len(marker)
                    date = line[pos:pos + 10].decode('ascii')
                    if date in dates:
                        row = json.loads(line)
                        pair = row['ticker'], date
                        assert pair not in seen, pair
                        seen.add(pair)
                        limits.append([*pair, row['raw_close'], row['ceiling_price'], row['floor_price']])
            actual = digest.hexdigest()
        else:
            actual = sha(path)
        assert before == fingerprint(path), ('Input bị thay đổi khi đọc', path)
        assert actual == expected[key], key
        hashes[key] = dict(path=str(path.relative_to(ROOT)).replace('\\', '/'), sha256=actual, **before)
    limits_path = output / 'verified_limits.json'
    write_json(limits_path, limits)
    bpit = ROOT / config['financial_pit']['artifact']
    result = dict(status='PASS', created_at=datetime.now(timezone.utc).isoformat(), identity=identity,
                  sources_sha256=sha(ROOT / 'M3/sources.json'), inputs=hashes,
                  bpit_metadata={name: sha(bpit / name) for name in ['manifest.json', 'models.json', 'audit.json']},
                  limits=dict(path=str(limits_path), sha256=sha(limits_path), dates=sorted(dates), rows=len(limits)))
    write_json(output / 'preflight.json', result)
    return result

In [4]:
class Context:

    def __init__(self):
        self.config, self.runs = validate_runs()
        self.identity = source_identity()
        self.output = Path(os.environ.get('M3C_OUTPUT_DIR', str(ROOT / 'M3/reproduction' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')))).resolve()
        self.output.mkdir(parents=True, exist_ok=True)
        proof = self.output / 'preflight.json'
        if not proof.exists():
            preflight(self.output)
        self.proof = read_json(proof)
        assert self.proof['status'] == 'PASS' and self.proof['sources_sha256'] == sha(ROOT / 'M3/sources.json')
        assert self.proof['identity']['module_hashes'] == self.identity['module_hashes']
        for key, path in input_paths(self.config).items():
            record = self.proof['inputs'][key]
            assert fingerprint(path) == {k: record[k] for k in ['size', 'mtime_ns']}, ('Cần chạy lại preflight', path)
            assert record['sha256'] == self.runs['development']['manifest']['inputs_sha256'][key]
        bpit = ROOT / self.config['financial_pit']['artifact']
        for name, digest in self.proof['bpit_metadata'].items():
            assert sha(bpit / name) == digest
        self.market = inputs.market_snapshots()
        self.availability = inputs.availability()
        self.indicators = inputs.indicators()
        self.sectors = inputs.sectors()
        financial, order, _, cutoffs = inputs.bpit(bpit, include_cutoffs=True)
        self.cutoffs = cutoffs
        self.strategies = Strategies(self.config, self.market, financial, self.availability, self.sectors, self.indicators, fin_order=order, financial_cutoffs=cutoffs)
        self.calendar = inputs.calendar()

    def read(self, segment, kind, strategy=None):
        name = f'{kind}_{strategy}.csv' if strategy else f'{kind}.csv'
        return csv_rows(self.runs[segment]['folder'] / name)

    def decisions(self, segment):
        m = self.runs[segment]['manifest']
        return sorted((d for d in self.market if m['segment_start'] <= d <= m['segment_end_date']))

    def execution(self, decision):
        i = bisect_right(self.calendar, decision)
        return self.calendar[i] if i < len(self.calendar) else ''

    def provenance(self):
        return [dict(input=key, **row) for key, row in self.proof['inputs'].items()]

    def target_rows(self, segment, strategy, date, result=None):
        result = result or self.strategies.target_for(strategy, date)
        execution = self.execution(date)
        status = result.get('status', 'OK')
        if not execution or execution > self.runs[segment]['manifest']['segment_end_date']:
            status = 'NOT_EXECUTED_SEGMENT_END'
        target = result.get('target') or {}
        rows = [dict(decision_date=date, execution_date=execution, ticker=t, rank=i + 1, target_weight=target[t], candidates=result['candidates'], status=status) for i, t in enumerate(result['picks'])]
        if not rows:
            rows = [dict(decision_date=date, execution_date=execution, ticker='', rank='', target_weight='', candidates=result['candidates'], status=status)]
        return (rows, result)

    def finish(self):
        for key, path in input_paths(self.config).items():
            record = self.proof['inputs'][key]
            assert fingerprint(path) == {k: record[k] for k in ['size', 'mtime_ns']}, ('Input thay đổi trong phiên', path)
        bpit = ROOT / self.config['financial_pit']['artifact']
        for name, digest in self.proof['bpit_metadata'].items():
            assert sha(bpit / name) == digest
        for segment in self.runs:
            seal = self.runs[segment]['seal']
            for name, digest in seal['artifacts_sha256'].items():
                assert sha(self.runs[segment]['folder'] / name) == digest
        source_identity()
        return 'PASS — nguồn canonical giữ nguyên; không có Final performance.'

In [5]:
ctx = Context()
print('Code nguồn:', ctx.identity['source_commit'])
print('Code thực thi:', ctx.identity['execution_commit'])
print('Python:', ctx.identity['executable'])
print('Config và nguồn canonical: PASS')
print('Evidence:', ctx.output)

Code nguồn: c9c1f5f0c88f2dbb3ebbb50effa47003b24b66cf
Code thực thi: c9c1f5f0c88f2dbb3ebbb50effa47003b24b66cf
Python: C:\Users\HP\Downloads\Phân cụm động lượng TTCK\.tmp\m3c-kernel\Scripts\python.exe
Config và nguồn canonical: PASS
Evidence: C:\Users\HP\Downloads\Phân cụm động lượng TTCK\M3\reproduction\20261104T154454296717Z-direct-notebooks


## 1. Identity và hash đầu vào

Runner hash đầy đủ bảy input một lần trong phiên, đồng thời lưu snapshot trần/sàn đã kiểm chứng. Mỗi kernel kiểm tra fingerprint để phát hiện input thay đổi; mở notebook riêng không có evidence sẽ chạy preflight mới. Hai seal và đủ 33 file mỗi run được kiểm tra lại. B-PIT metadata hiện hành cũng được ghi hash; không build B-PIT lại.

In [6]:
table(ctx.provenance(), ["input", "path", "sha256", "size"])
table([dict(segment=s, run_id=v["manifest"]["run_id"], **v["manifest"]["counts"]["combined"]) for s,v in ctx.runs.items()])

input,path,sha256,size
terminal_sha256,data/web/delta-intelligence-v4/terminal.sqlite,0662585b95a7d529854dbcea8816e47e24466dc55ff1368cbe923c018c5670e9,526651392
indicators_sha256,data/financial/indicators_v1/results/indicators-2c4a24e3830d325f/indicators.sqlite,26bfc2d922a8f39f1fb8234e6b50c788ef83ac4b229059ed339a14ebd1213023,111276032
availability_sha256,data/financial/disclosure_dates_v1/availability-4f0f83108e761dae/availability.csv,4f0f83108e761daeb86289e290de3aff0379abbf1ed630ac31c842146186648f,873745
c8_benchmark_sha256,artifacts/cafef_primary/cafef-c8-complete-only-v2/canonical/benchmark_daily.jsonl,75fc214cb079159fe2df03f29fe9c8075705b24cbb114f09f0e800bf3c4285a2,954587
c8_calendar_sha256,artifacts/cafef_primary/cafef-c8-complete-only-v2/canonical/trading_calendar.jsonl,9aa4ffd869052654dc5b9ed7f6b19c8339469057fa80dda2a66f9a6aecb68b6f,2864640
c8_prices_sha256,artifacts/cafef_primary/cafef-c8-complete-only-v2/canonical/prices_daily.jsonl,20ecee90df48869fa4ed7196c64ff857775ebd54fcaeb1b93ced76d6635a2b1f,4193211376
b_pit_labels_sha256,data/financial/b_pit_v1/bpit-4a9d2e87f0de84ec/labels.csv,4a9d2e87f0de84ec9e80364b24043cb6091aa4f34f7dcfd187d8526e9f44579a,225357


segment,run_id,decisions,executed,too_small,not_executed_segment_end
development,20261104T121107Z-5557288c,60,59,13,1
holdout,20261104T133507Z-5557288c,24,23,1,1


## 2. Tái lập PIT trên một số decision dates

Dùng `Strategies.target_for` và `universe.financial_state` chính thức, kiểm tra FY mới nhất usable, stale và model cutoff. Bảng minh họa lấy tối đa ba ứng viên mỗi ngày; assertion kiểm tra toàn bộ ứng viên trong các ngày mẫu. Target mẫu đối chiếu cả mã, rank, trọng số và status với CSV canonical.

FY trước 2022 dùng nguồn LAG_RULE, từ FY2022 dùng CAFEF_DISCLOSURE theo policy đã khóa. Ngày usable là phiên kế tiếp sau ngày công bố, không suy giờ.

### Kiểm tra PIT bằng module chính thức

In [7]:
def pit_samples(ctx):
    checks = []
    counts = []
    for segment in ctx.runs:
        dates = ctx.decisions(segment)
        examples = sorted({dates[0], dates[len(dates) // 2], dates[-1], *[d for d in dates if d.endswith(('04-28', '04-29', '04-30'))]})
        for decision in examples:
            result = ctx.strategies.target_for('combined', decision)
            generated, _ = ctx.target_rows(segment, 'combined', decision, result)
            original = [r for r in ctx.read(segment, 'targets', 'combined') if r['decision_date'] == decision]
            compare_rows(generated, original, ('PIT mẫu', segment, decision))
            count = 0
            for row in result['audit']:
                if row['in_candidate_universe']:
                    year = int(row['financial_year'])
                    usable = ctx.availability[row['ticker'], year]['usable_from_date']
                    chosen, reason = universe.financial_state(row['ticker'], decision, ctx.availability, sorted({y for _, y in ctx.availability}), ctx.config['stale_months'])
                    assert year == chosen and reason is None
                    assert usable <= decision and ctx.cutoffs[year] <= decision
                    if count < 3:
                        checks.append(dict(segment=segment, decision_date=decision, ticker=row['ticker'], financial_year=year, usable_from_date=usable, cutoff=ctx.cutoffs[year], result='PASS'))
                    count += 1
            counts.append(dict(segment=segment, decision_date=decision, candidates=result['candidates'], checked_eligible_rows=count, target='PASS'))
    write_json(ctx.output / '01_verification.json', dict(status='PASS', samples=checks, checks=counts, provenance=ctx.provenance(), identity=ctx.identity))
    return (checks, counts)

In [8]:
samples, checks = pit_samples(ctx)
table(checks)
table(samples)

segment,decision_date,candidates,checked_eligible_rows,target
development,2019-01-31,63,63,PASS
development,2020-04-29,6,6,PASS
development,2021-04-29,30,30,PASS
development,2021-07-30,11,11,PASS
development,2022-04-29,3,3,PASS
development,2023-04-28,12,12,PASS
development,2023-12-29,80,80,PASS
holdout,2024-01-31,97,97,PASS
holdout,2025-01-24,12,12,PASS
holdout,2025-04-29,4,4,PASS


segment,decision_date,ticker,financial_year,usable_from_date,cutoff,result
development,2019-01-31,AGM,2017,2018-07-02,2018-07-02,PASS
development,2019-01-31,AMV,2017,2018-07-02,2018-07-02,PASS
development,2019-01-31,CAP,2017,2018-07-02,2018-07-02,PASS
development,2020-04-29,D11,2018,2019-07-01,2019-07-01,PASS
development,2020-04-29,GVT,2018,2019-07-01,2019-07-01,PASS
development,2020-04-29,IDV,2018,2019-07-01,2019-07-01,PASS
development,2021-04-29,BVG,2019,2020-07-01,2020-07-01,PASS
development,2021-04-29,CMT,2019,2020-07-01,2020-07-01,PASS
development,2021-04-29,DGW,2019,2020-07-01,2020-07-01,PASS
development,2021-07-30,CBI,2020,2021-07-01,2021-07-01,PASS


## 3. Giới hạn của kết luận PIT

PASS chứng minh việc dùng availability/cutoff đúng theo snapshot đã pin. Không xác nhận dữ liệu tài chính là vintage nguyên thủy hoặc không có restatement. Các giới hạn VCG/SBT được owner chấp nhận vẫn áp dụng; không mở lại audit PDF trong M3-C.

In [9]:
print(ctx.finish())
print("NOTEBOOK_01_PASS")

PASS — nguồn canonical giữ nguyên; không có Final performance.
NOTEBOOK_01_PASS


In [10]:
write_json(ctx.output / 'direct_01_done.json', dict(status='PASS', notebook='01_Du_Lieu_Va_PIT.ipynb', code_sha256=notebook_code_sha256(ROOT / 'M3/notebooks/01_Du_Lieu_Va_PIT.ipynb')))
print('DIRECT_NOTEBOOK_01_PASS')

DIRECT_NOTEBOOK_01_PASS
